In [1]:
# Step 1: Imports
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import csv
from typing import List


# Step 2: Load CSV Function
def load_faq_csv(path: str) -> List[Document]:

    docs = []

    with open(path, "r", encoding="utf-8") as f:

        reader = csv.DictReader(f)

        for row in reader:

            q = row["question"].strip()
            a = row["answer"].strip()

            docs.append(
                Document(
                    page_content=f"Q: {q}\nA: {a}"
                )
            )

    return docs


# Step 3: Load CSV
docs = load_faq_csv("./bank_faq.csv")


# Step 4: Split Documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0
)

chunks = splitter.split_documents(docs)


# Step 5: Create Embeddings
emb = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)


# Step 6: Create FAISS Vector Store
store = FAISS.from_documents(
    chunks,
    emb
)




In [3]:
# Step 7: Ask Question
query = "How do I reset my mobile banking password?"


# Step 8: Perform Similarity Search
results = store.similarity_search(
    query,
    k=3
)


# Step 9: Print Results
for i, r in enumerate(results):

    print(f"\nResult {i+1}")
    print("-" * 50)

    print(r.page_content)


Result 1
--------------------------------------------------
Q: How do I reset my mobile banking password?
A: To reset your password, open the NeoBank app, tap 'Forgot Password' on the login screen, and verify your identity using the OTP sent to your registered mobile number.

Result 2
--------------------------------------------------
Q: How do I update my mailing address?
A: Log in to online banking, go to 'Profile & Settings', select 'Personal Details', and update your address. You may need to upload a proof of address document.

Result 3
--------------------------------------------------
Q: I lost my credit card. What should I do?
A: Immediately block your card via the mobile app under 'Card Settings' or call our 24/7 fraud hotline at 1-800-NEO-BANK. We will issue a replacement within 3-5 business days.


In [4]:
# Step 1: Imports
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Step 2: Sample Documents
docs = [
    Document(page_content="ABC Phones offers prepaid plans"),
    Document(page_content="Roaming can be activated from settings"),
    Document(page_content="5G plans start from ₹399")
]

# Step 3: Create Embeddings
embeddings = FastEmbedEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# Step 4: Create FAISS Vector DB
store = FAISS.from_documents(
    docs,
    embeddings
)

# Step 5: Ask Question
query = "How to activate roaming?"

# Step 6: Semantic Search
results = store.similarity_search(query, k=2)

# Step 7: Print Results
for r in results:
    print(r.page_content)

Roaming can be activated from settings
5G plans start from ₹399


#### Understading LLM response

In [5]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

In [6]:
abc=[HumanMessage(content='What is the daily withdrawal limit for the Gold Debit Card?', additional_kwargs={}, response_metadata={}, id='50f8149a-3227-4a75-a500-e372ca40eae1'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: daily withdrawal limit for Gold Debit Card. Likely from FAQ. Use search_faq.', 'tool_calls': [{'id': 'fc_2bbf9f3d-4bdb-4b16-8e2d-e036c1b08a67', 'function': {'arguments': '{"query":"daily withdrawal limit Gold Debit Card"}', 'name': 'search_faq'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 236, 'total_tokens': 289, 'completion_time': 0.059193904, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.011479211, 'prompt_tokens_details': None, 'queue_time': 0.031833095, 'total_time': 0.070673115}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3192-e34c-78d2-8528-6105998d371e-0', tool_calls=[{'name': 'search_faq', 'args': {'query': 'daily withdrawal limit Gold Debit Card'}, 'id': 'fc_2bbf9f3d-4bdb-4b16-8e2d-e036c1b08a67', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 53, 'total_tokens': 289, 'output_token_details': {'reasoning': 24}}), ToolMessage(content="Found 3 relevant FAQ entries:\n\nFAQ Entry 1:\nQ: What is the daily withdrawal limit for the Gold Debit Card?\nA: The daily ATM withdrawal limit for the Gold Debit Card is $2,000. Point-of-sale transaction limit is $5,000 per day.\n\n---\n\nFAQ Entry 2:\nQ: I lost my credit card. What should I do?\nA: Immediately block your card via the mobile app under 'Card Settings' or call our 24/7 fraud hotline at 1-800-NEO-BANK. We will issue a replacement within 3-5 business days.\n\n---\n\nFAQ Entry 3:\nQ: Is my money safe with NeoBank?\nA: Yes, all deposits are insured up to $250,000 by the FDIC. We also use 256-bit encryption to protect your digital data.", name='search_faq', id='ff35ec91-2c01-4b03-b9af-fe3504051183', tool_call_id='fc_2bbf9f3d-4bdb-4b16-8e2d-e036c1b08a67'), AIMessage(content='The Gold Debit Card allows you to withdraw up to **$2,000 per day** from ATMs.  \n(For reference, the point‑of‑sale transaction limit is $5,000 per day.)', additional_kwargs={'reasoning_content': 'We have the answer: daily ATM withdrawal limit $2,000. Also point-of-sale transaction limit $5,000 per day. The user asked: "What is the daily withdrawal limit for the Gold Debit Card?" So answer: $2,000 daily ATM withdrawal limit. Provide also mention of POS limit if relevant.'}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 441, 'total_tokens': 559, 'completion_time': 0.134060965, 'completion_tokens_details': {'reasoning_tokens': 66}, 'prompt_time': 0.045924752, 'prompt_tokens_details': None, 'queue_time': 0.032484288, 'total_time': 0.179985717}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e1634aab79', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3192-e416-78d2-9e83-9b888719196a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 441, 'output_tokens': 118, 'total_tokens': 559, 'output_token_details': {'reasoning': 66}})]


In [7]:
abc

[HumanMessage(content='What is the daily withdrawal limit for the Gold Debit Card?', additional_kwargs={}, response_metadata={}, id='50f8149a-3227-4a75-a500-e372ca40eae1'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer: daily withdrawal limit for Gold Debit Card. Likely from FAQ. Use search_faq.', 'tool_calls': [{'id': 'fc_2bbf9f3d-4bdb-4b16-8e2d-e036c1b08a67', 'function': {'arguments': '{"query":"daily withdrawal limit Gold Debit Card"}', 'name': 'search_faq'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 236, 'total_tokens': 289, 'completion_time': 0.059193904, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.011479211, 'prompt_tokens_details': None, 'queue_time': 0.031833095, 'total_time': 0.070673115}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_976a85d17a', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider':

In [8]:
abc[1].additional_kwargs['tool_calls'][0]['function']['name']

'search_faq'